## 1. Load environment variables
Reads the `.env` file so `OPENAI_API_KEY` is available to the OpenAI and LangChain libraries. Nothing is printed, so the key never ends up in the notebook output.

**Expected result:** `True` (the `.env` file was found).

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 2. Check the API key and LangChain version
Confirms that the key was loaded (prints only `True`/`False`, never the key itself) and that this kernel runs the pinned LangChain version.

**Expected result:** `API key loaded: True` and `langchain: 0.2.17`. Any other version means the wrong kernel is selected.

In [2]:
import os
import langchain

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)
print("langchain:", langchain.__version__)

API key loaded: True
langchain: 0.2.17


## 3. Check the required libraries
Tries to import each package the project needs and reports which ones are installed.

**Expected result:** `OK` for `langchain_community`, `langchain_openai` and `chromadb`. `pypdf` is missing at first and is installed in the next step. `langchain_chroma` is not needed: we use the Chroma class from `langchain_community`.

In [3]:
import importlib

for pkg in ["langchain_community", "langchain_openai", "langchain_chroma", "chromadb", "pypdf"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"OK       {pkg} {getattr(mod, '__version__', '')}")
    except ImportError:
        print(f"MISSING  {pkg}")

OK       langchain_community 0.2.19
OK       langchain_openai 
MISSING  langchain_chroma
OK       chromadb 1.5.9
OK       pypdf 6.21.4


## 4. Install `pypdf`
Installs the PDF reader into the kernel that is running this notebook (`%pip` targets the active kernel).

**Expected result:** `Successfully installed pypdf`. A kernel restart may be suggested; if so, re-run cell 1 afterwards.

In [4]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


## 5. Smoke test: Chroma vector store
Creates a tiny in-memory Chroma database with two fake embeddings, only to prove that Chroma works with the installed `chromadb` version before building anything on top of it. No API calls, no cost.

**Expected result:** `Chroma OK, docs: 2` (a deprecation warning is harmless).

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import FakeEmbeddings

test_db = Chroma.from_texts(["hello", "world"], FakeEmbeddings(size=8))
print("Chroma OK, docs:", test_db._collection.count())

Chroma OK, docs: 2


## 6. Inspect the internal documents
Read-only scan of the 20 internal PDFs: prints the page count and size of each one and searches for words such as version, effective date, last updated or superseded. Used to find out how the documents are structured and whether some may be outdated or duplicated.

**Expected result:** one line per PDF plus a few matching snippets. Findings: every document has a header with ID, version and review date; several include a version history; `CS-POL-014` appears in two different documents.

In [6]:
import re
from pathlib import Path
from pypdf import PdfReader

INTERNAL = Path("documents/internal_docs_by_area")
pattern = re.compile(r"(version|effective date|last updated|revised|supersed\w*|deprecated|outdated|draft)[^\n]{0,60}", re.I)

for pdf in sorted(INTERNAL.rglob("*.pdf")):
    reader = PdfReader(pdf)
    text = "\n".join(p.extract_text() or "" for p in reader.pages)
    hits = [m.group(0).strip() for m in pattern.finditer(text)][:4]
    print(f"{pdf.parent.name}/{pdf.name} | {len(reader.pages)} pp | {len(text)} chars")
    for h in hits:
        print("    ·", h)

Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf | 7 pp | 12094 chars
    · Version: 3.2
    · Version 3.2: Added P1/P2 response targets, updated refund approval
    · Version 3.1: Added executive complaint handling and Security Hub re
    · Version 3.0: Consolidated country escalation procedures and introdu
Customer_support/IronStore_Customer_Support_Knowledge_Base_Style_Guide.pdf | 7 pp | 11810 chars
    · outdated images, or missing translations are content defects and mus
    · draft and records the source of each policy or operational claim.
    · Drafts, maintains, and reviews assigned articles.
Customer_support/IronStore_Refund_and_Complaint_Handling_Policy.pdf | 7 pp | 13552 chars
    · Version: 3.2
    · revised date.
    · revised approval
Deliveries/IronStore_Returns_and_Reverse_Logistics_Procedure.pdf | 6 pp | 12402 chars
    · Version: 3.2
    · revised refund ageing targets.
Deliveries/IronStore_Shipping_and_Delivery_SLA_Policy.pdf | 6 pp | 11734 char

## 7. Load the internal PDFs and split them into chunks
Loads the 20 internal PDFs page by page with `PyPDFLoader` (which keeps the file path and page number), adds metadata to each page (`area`, `doc_title`, `doc_type`, 1-based `page`) and splits the text into chunks of about 1000 characters with 150 characters of overlap. Overlap prevents a sentence from being cut exactly at a chunk boundary.

**Expected result:** `pages: 127 | chunks: 341`, followed by the metadata and the first characters of one sample chunk.

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

docs = []
for pdf in sorted(INTERNAL.rglob("*.pdf")):
    pages = PyPDFLoader(str(pdf)).load()
    for p in pages:
        p.metadata["area"] = pdf.parent.name
        p.metadata["doc_title"] = pdf.stem.replace("IronStore_", "").replace("_", " ")
        p.metadata["doc_type"] = "internal"
        p.metadata["page"] = p.metadata["page"] + 1   # 1-based for humans
    docs.extend(pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print("pages:", len(docs), "| chunks:", len(chunks))
print(chunks[10].metadata)
print(chunks[10].page_content[:300])

pages: 127 | chunks: 341
{'source': 'documents/internal_docs_by_area/Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf', 'page': 5, 'area': 'Customer_support', 'doc_title': 'Customer Support Escalation Procedures', 'doc_type': 'internal'}
5.5 Complaints, legal, and regulatory matters
Escalate to Customer Advocacy when:
  The customer has made two unsuccessful contacts about the same issue.
  The customer alleges discrimination, harassment, or unreasonable treatment.
  A formal complaint is requested.
  The customer references a r


## 8. Add the document header data to every chunk
Each PDF states its document ID, version and review date in the header of page 1. This cell extracts those three values with a regular expression and copies them into the metadata of every chunk of that document, so answers can cite the exact version and overlapping documents can be detected later. Missing values are stored as `n/a`.

**Expected result:** a sample chunk whose metadata now also contains `doc_id`, `version` and `last_reviewed`.

In [8]:
import re

def header_field(text, label):
    m = re.search(rf"{label}:\s*([^\n|]+)", text)
    return m.group(1).strip() if m else None

header_by_source = {}
for d in docs:
    if d.metadata["page"] == 1:
        t = d.page_content
        header_by_source[d.metadata["source"]] = {
            "doc_id": header_field(t, "Document ID") or header_field(t, "Policy ID") or header_field(t, "Policy reference"),
            "version": header_field(t, "Version"),
            "last_reviewed": header_field(t, "Last reviewed"),
        }

for c in chunks:
    for k, v in header_by_source[c.metadata["source"]].items():
        c.metadata[k] = v if v else "n/a"

print(chunks[10].metadata)

{'source': 'documents/internal_docs_by_area/Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf', 'page': 5, 'area': 'Customer_support', 'doc_title': 'Customer Support Escalation Procedures', 'doc_type': 'internal', 'doc_id': 'CS-POL-014', 'version': '3.2', 'last_reviewed': 'August 2026'}


## 9. Create embeddings and store the chunks in Chroma
Converts every chunk into an embedding (a vector of numbers that represents its meaning) with `text-embedding-3-small` and saves them, with their metadata, in a persistent Chroma collection called `internal_docs` (folder `chroma_db`). The collection is deleted first, so re-running the cell never duplicates chunks. A similarity search with a test question checks that retrieval works.

**Expected result:** `chunks indexed: 341` and three sources, the first ones from the Leave and Absence Policy. The `LangChainDeprecationWarning` about `Chroma` is harmless with LangChain 0.2.17.

In [9]:
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")

vectordb = Chroma(collection_name="internal_docs", persist_directory="chroma_db", embedding_function=emb)
vectordb.delete_collection()   # so re-running never duplicates chunks
vectordb = Chroma.from_documents(chunks, emb, collection_name="internal_docs", persist_directory="chroma_db")

print("chunks indexed:", vectordb._collection.count())

for r in vectordb.similarity_search("How many days of annual leave do employees get?", k=3):
    print("-", r.metadata["doc_title"], "| page", r.metadata["page"])

/var/folders/5t/lmqvg1xx4nl0pnr06vf0vy2h0000gn/T/ipykernel_77875/3318230371.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectordb = Chroma(collection_name="internal_docs", persist_directory="chroma_db", embedding_function=emb)


chunks indexed: 341
- Leave and Absence Policy | page 2
- Leave and Absence Policy | page 2
- Leave and Absence Policy | page 1


## 10. Imports and language models
Imports the LangChain pieces used from here on and creates the two chat models: `gpt-4o-mini` (cheap, for plain product lookups) and `gpt-4o` (stronger, for policy questions where several documents may disagree). Both use temperature 0 so answers are stable.

**Expected result:** no output.

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)        # cheap model
strong_llm = ChatOpenAI(model="gpt-4o", temperature=0)      # stronger model for policy questions

## 11. Check the extracted header data
Read-only check: prints the document ID, version and review date extracted from the header of each internal PDF, to confirm that the extraction in step 8 worked for all 20 documents.

**Expected result:** 20 lines. Documents without a version in their header show `None` (stored as `n/a`). `CS-POL-014` appears twice: it is the ID of two different documents.

In [11]:
for src, meta in header_by_source.items():
    print(Path(src).stem.replace("IronStore_", "")[:45].ljust(46), meta)

Customer_Support_Escalation_Procedures         {'doc_id': 'CS-POL-014', 'version': '3.2', 'last_reviewed': 'August 2026'}
Customer_Support_Knowledge_Base_Style_Guide    {'doc_id': 'CS-KB-STD-004', 'version': None, 'last_reviewed': 'August 2026'}
Refund_and_Complaint_Handling_Policy           {'doc_id': 'CS-POL-014', 'version': '3.2', 'last_reviewed': 'August 2026'}
Returns_and_Reverse_Logistics_Procedure        {'doc_id': 'DEL-RET-004', 'version': '3.2', 'last_reviewed': 'August 2026'}
Shipping_and_Delivery_SLA_Policy               {'doc_id': 'DEL-SLA-004', 'version': None, 'last_reviewed': 'August 2026'}
Warehouse_Operations_Manual                    {'doc_id': 'DEL-WH-001', 'version': '3.2', 'last_reviewed': 'August 2026'}
Budget_Approval_Workflow                       {'doc_id': 'FIN-PR-014', 'version': None, 'last_reviewed': 'August 2026'}
Expense_Reimbursement_Policy                   {'doc_id': 'FIN-EXP-004', 'version': None, 'last_reviewed': 'August 2026'}
Payroll_Processing_Pro

## 12. Index the product catalog (second collection)
Does for the 63 catalog PDFs what steps 7 to 9 did for the internal documents, in its own Chroma collection (`product_catalog`). Differences: chunks of 1500 characters (a product entry is about 1000 characters), the printed "Page N" labels are rewritten as "Section:" so they cannot be mistaken for PDF page numbers, and each catalog is marked `current` or `outdated` from its review date. The 2021 catalog is detected as outdated.

**Expected result:** `catalog pages: 2181 | chunks: 2229`, `outdated catalogs: {'Product Catalog - IronStore Old Product Catalog 2021'}` and `catalog chunks indexed: 2229`. **Re-run the next cell after this one**, because it recreates the collection.

In [12]:
def last_reviewed_of(text):
    m = re.search(r"Last reviewed(?: in)?:?\s*([A-Za-z]+ \d{4})", text)
    return m.group(1) if m else "n/a"

catalog_docs = []
for pdf in sorted(Path("documents/product_catalog").glob("*.pdf")):
    pages = PyPDFLoader(str(pdf)).load()
    first = pages[0].page_content
    category = pdf.stem.replace("IronStore_Product_Catalog_", "").replace("_", " ")
    edition = header_field(first, "Internal catalog edition") or "n/a"
    reviewed = last_reviewed_of(first)
    status = "outdated" if reviewed != "n/a" and int(reviewed[-4:]) < 2026 else "current"
    for p in pages:
        p.metadata["area"] = "product_catalog"
        p.metadata["doc_title"] = f"Product Catalog - {category}"
        p.metadata["doc_type"] = "catalog"
        p.metadata["doc_id"] = edition
        p.metadata["version"] = "n/a"
        p.metadata["last_reviewed"] = reviewed
        p.metadata["status"] = status
        p.metadata["page"] = p.metadata["page"] + 1   # physical PDF page
        p.page_content = re.sub(r"(?m)^Page \d+ [—-] ", "Section: ", p.page_content)
        p.page_content = f"[Catalog: {category}]\n" + p.page_content
    catalog_docs.extend(pages)

cat_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
catalog_chunks = cat_splitter.split_documents(catalog_docs)
print("catalog pages:", len(catalog_docs), "| chunks:", len(catalog_chunks))
print("outdated catalogs:", {c.metadata["doc_title"] for c in catalog_chunks if c.metadata["status"] == "outdated"})

catalog_db = Chroma(collection_name="product_catalog", persist_directory="chroma_db", embedding_function=emb)
catalog_db.delete_collection()
catalog_db = Chroma.from_documents(catalog_chunks, emb, collection_name="product_catalog", persist_directory="chroma_db")
print("catalog chunks indexed:", catalog_db._collection.count())

catalog pages: 2181 | chunks: 2229
outdated catalogs: {'Product Catalog - IronStore Old Product Catalog 2021'}
catalog chunks indexed: 2229


## 13. Create the two retrievers
A retriever returns the chunks closest in meaning to a question. The internal one returns 6 chunks; the catalog one returns 4 and only searches catalogs marked `current`, so the 2021 prices are never used.

**Expected result:** no output.

In [13]:
internal_retriever = vectordb.as_retriever(search_kwargs={"k": 6})
catalog_retriever = catalog_db.as_retriever(search_kwargs={"k": 4, "filter": {"status": "current"}})

## 14. Diagnostic: which refund rules reach the model?
Read-only diagnostic used to find out whether a wrong answer came from retrieval or from the model. It prints, for each retrieved chunk, the lines that contain a euro amount. For the refund question it showed that all the relevant rules were retrieved, so the weak point was the model, not the search.

**Expected result:** a list of chunks with their approval thresholds from the Returns Procedure, the Refund Policy and the Escalation Procedures.

In [14]:
q = "Who can approve a refund above €1,000?"
for d in internal_retriever.invoke(q):
    lines = [l.strip() for l in d.page_content.splitlines() if "€" in l]
    print(f"{d.metadata['doc_title']} | p.{d.metadata['page']}")
    for l in lines:
        print("     ", l[:140])

Returns and Reverse Logistics Procedure | p.4
        Up to €250: Returns Team Lead may approve refund, replacement, or partial refund.
        €250–€1,000: Returns Team Lead approval plus Finance Operations review for non-standard outcomes.
        Above €1,000: Deliveries Manager and Finance Operations approval.
Refund and Complaint Handling Policy | p.3
        Up to €150: trained agent within documented authority.
        €150.01–€500: Customer Support Team Lead.
        Above €500: Customer Operations Manager and Finance Operations.
        Any goodwill amount above €50: Team Lead approval.
Refund and Complaint Handling Policy | p.3
        Up to €25: agent may refund or replace without approval when evidence is sufficient.
        €25.01–€150: team-lead approval is required for a no-return refund unless an automatic rule applies.
        Over €150: Returns Operations or Payments Operations approval is required.
        Up to €150: trained agent within documented author

## 15. Context grouped by document and the final prompt
`format_docs_grouped` presents the retrieved chunks grouped under one header per document, which makes the model attribute each rule to the right document. `prompt_v2` tells the model to answer only from the context, to cite `[document title, p. X]`, to use the page from the source header and not the printed "Page N" label, to go through every relevant document, and to start with "The documents differ:" when documents give different rules. If nothing relevant is found it must reply `I could not find this in the IronStore documents.`

**Expected result:** no output.

In [15]:
from collections import defaultdict

def format_docs_grouped(docs):
    groups = defaultdict(list)
    for d in docs:
        m = d.metadata
        groups[(m["doc_title"], m["doc_id"], m["version"])].append(d)
    parts = []
    for (title, doc_id, ver), ds in groups.items():
        v = f", v{ver}" if ver != "n/a" else ""
        parts.append(f"=== DOCUMENT: {title} ({doc_id}{v}) ===")
        for d in sorted(ds, key=lambda x: x.metadata["page"]):
            parts.append(f"[p. {d.metadata['page']}]\n{d.page_content}")
    return "\n\n".join(parts)

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system",
     "You are IronStore's internal assistant. Answer ONLY with the context below, which is grouped by "
     "document: each group starts with '=== DOCUMENT: title (id, version) ===' and every page block starts "
     "with its marker [p. X]. "
     "CITATIONS: cite every claim as [document title, p. X]. Take the title from the group header and the page X "
     "from the marker of the exact block that contains the fact (for example the block with the price or the rule), "
     "not from another block of the same document. Ignore any 'Page N' or 'Section' label written inside the text. "
     "NO ANSWER: only if the documents do not discuss the topic at all, reply exactly: "
     "'I could not find this in the IronStore documents.' "
     "PARTIAL ANSWER: if the documents discuss the topic but do not state the exact value asked for, do NOT "
     "use that sentence. Instead say what the documents state, what is missing, and where they say it can be "
     "found. Example: asked for a limit that a document says is set in each employee's contract, answer that "
     "the documents do not give a single figure and that it is set in the contract, citing the document. "
     "Documents contain a version history: apply the current rules, not older versions listed there. "
     "ANSWER FORMAT: (1) one or two sentences with the answer. "
     "(2) If more than one document addresses the question, go through EVERY document group in the context "
     "that is relevant and add one line each: '- <document title>: <what it says> [title, p. X]'. "
     "(3) If those documents give different rules, thresholds or approvers for the same case, "
     "start the answer with 'The documents differ:' and do not choose one of them."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

## 16. Router and final assistant
`ask(question)` first classifies the question with the cheap model as `policy`, `product` or `both`, then searches only the needed collection(s) and answers with `gpt-4o-mini` for product lookups or `gpt-4o` for policy questions. If the router returns anything unexpected it searches everything. It prints the route, the model used, the answer and the retrieved sources.

**Expected result:** no output (the function is only defined here). It is used by the evaluation cell below.

In [16]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Classify a question about IronStore. Reply with exactly one word: "
     "'policy' (internal policies and procedures: HR, finance, IT, legal, sales, support, deliveries, approvals), "
     "'product' (products, prices, availability, SKUs, catalog) or 'both'."),
    ("human", "{question}"),
])
router = router_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

def ask(question):
    route = router.invoke({"question": question}).strip().lower()
    if route not in ("policy", "product", "both"):
        route = "both"                      # if the router is unsure, search everywhere
    docs = []
    if route in ("policy", "both"):
        docs += internal_retriever.invoke(question)
    if route in ("product", "both"):
        docs += catalog_retriever.invoke(question)
    model = strong_llm   # gpt-4o-mini cited the wrong page with grouped context; the router only decides where to search
    answer = (prompt_v2 | model | StrOutputParser()).invoke(
        {"context": format_docs_grouped(docs), "question": question}
    )
    print(f"Q: {question}\n[route: {route} | model: {model.model_name}]\n")
    print(answer, "\n")
    print("Sources retrieved:")
    for d in docs:
        print(" -", d.metadata["doc_title"], "| p.", d.metadata["page"])


## 17. Evaluation: 16 test questions
Runs `ask()` on 16 questions whose answers are known: 9 single-document policy questions, 2 catalog lookups, 3 traps (a price that only exists in the outdated 2021 catalog, the conflicting refund approvers, and the annual-leave figure that the documents do not state) and 2 out-of-scope questions. Each answer is checked for the expected keywords (`must`) and for words that would signal an error (`must_not`).

**Expected result:** a PASS/FAIL line per question with the route chosen by the router, the score out of 16, and the full text of any failed answer. A FAIL needs a manual look: sometimes the answer is right and the check is too strict.

In [17]:
import io, contextlib

eval_set = [
    # answerable, single document
    {"q": "How far in advance must I request annual leave longer than five working days?", "must": ["20 calendar days"]},
    {"q": "What is the minimum password length for standard accounts?", "must": ["14"]},
    {"q": "Within how many days must expense claims be submitted in SpendHub?", "must": ["30"]},
    {"q": "Within how many hours should a reportable personal data breach be notified to the authority?", "must": ["72"]},
    {"q": "How long does domestic standard delivery take after dispatch?", "must": ["2", "4", "working days"]},
    {"q": "What commission rate applies to standard product categories?", "must": ["1.0"]},
    {"q": "What are the core hours for remote work?", "must": ["10:00", "15:00"]},
    {"q": "From what contract value does Legal review the agreement in ContractHub?", "must": ["100,000"]},
    {"q": "How long are customer service cases retained after closure?", "must": ["3 years"]},
    # product catalog
    {"q": "What is the price and availability of the Veltro VKT-1800 glass kettle?", "must": ["59", "in stock", "p. 21"]},
    {"q": "Is the Frigenta HTR-2000 Ceramic Fan Heater still available?", "must": ["discontinued"]},
    # traps
    {"q": "What is the price of the IronStore 24-inch Full HD Office Monitor (IS-ELEC-2101)?", "must": [], "must_not": ["149.99"]},   # only in the outdated 2021 catalog
    {"q": "Who can approve a refund above €1,000?", "must": ["differ", "Payments Operations", "Deliveries Manager"]},
    {"q": "How many days of annual leave do employees get?", "must": ["PeopleHub"], "must_not": ["25 days", "28 days", "30 days"]},
    # out of scope
    {"q": "What is the company's policy on cryptocurrency payments?", "must": ["could not find"]},
    {"q": "What is the capital of France?", "must": ["could not find"]},
]

rows = []
for t in eval_set:
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        ask(t["q"])
    out = buf.getvalue()
    route = re.search(r"\[route: (\w+)", out).group(1)
    answer = out.split("]\n\n", 1)[1].split("\n\nSources retrieved:")[0]
    low = answer.lower()
    ok = all(m.lower() in low for m in t["must"]) and not any(n.lower() in low for n in t.get("must_not", []))
    rows.append((ok, route, t["q"], answer))

for ok, route, q, _ in rows:
    print("PASS" if ok else "FAIL", "|", route.ljust(7), "|", q)
print("\nScore:", sum(r[0] for r in rows), "/", len(rows))

for ok, route, q, answer in rows:
    if not ok:
        print("\n--- FAILED:", q, "\n", answer)

PASS | policy  | How far in advance must I request annual leave longer than five working days?
PASS | policy  | What is the minimum password length for standard accounts?
PASS | policy  | Within how many days must expense claims be submitted in SpendHub?
PASS | policy  | Within how many hours should a reportable personal data breach be notified to the authority?
PASS | policy  | How long does domestic standard delivery take after dispatch?
PASS | policy  | What commission rate applies to standard product categories?
PASS | policy  | What are the core hours for remote work?
PASS | policy  | From what contract value does Legal review the agreement in ContractHub?
PASS | policy  | How long are customer service cases retained after closure?
PASS | product | What is the price and availability of the Veltro VKT-1800 glass kettle?
PASS | product | Is the Frigenta HTR-2000 Ceramic Fan Heater still available?
PASS | product | What is the price of the IronStore 24-inch Full HD Office Monitor (IS

## 18. Diagnostic: kettle citation
Read-only check used to debug a wrong page citation. It confirms that the new prompt is active and shows which retrieved catalog chunks contain the price `59.00`.

**Expected result:** `new prompt active: True` and four catalog chunks; the Home Appliances page 21 chunk contains `59.00`.

In [18]:
print("new prompt active:", "PARTIAL ANSWER" in prompt_v2.messages[0].prompt.template)

q = "What is the price and availability of the Veltro VKT-1800 glass kettle?"
for d in catalog_retriever.invoke(q):
    print(d.metadata["doc_title"], "| p.", d.metadata["page"], "| contains 59.00:", "59.00" in d.page_content)

new prompt active: True
Product Catalog - Home Appliances | p. 21 | contains 59.00: True
Product Catalog - Kitchen and Dining | p. 5 | contains 59.00: False
Product Catalog - Home Essentials | p. 2 | contains 59.00: True
Product Catalog - Home Appliances | p. 3 | contains 59.00: False
